In [31]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils

from matplotlib import pyplot as plt
from scipy.spatial.distance import mahalanobis


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [32]:
main_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))
main_df['canonical_casenum'] = main_df['title'].apply(lambda x: parse_casenum(x)['canonical_casenum'])
main_df = main_df.sort_values(by=['canonical_casenum', 'date', 'item_no'], ascending=True).reset_index(drop=True)
main_df['times_appeared'] = main_df.groupby('canonical_casenum').cumcount() + 1

main_df['atypicality_z'] = (main_df['atypicality'] - main_df['atypicality'].mean()) # demean 

In [33]:
df = []
for idx, row in main_df.iterrows():
    if idx==0:
        continue
    prev_row = main_df.iloc[idx-1]
    if prev_row['canonical_casenum'] != row['canonical_casenum']:
        continue
    days_between = (pd.to_datetime(row['date']) - pd.to_datetime(prev_row['date'])).days
    prev_members_present = set(list(prev_row['ayes']) + list(prev_row['noes']) + list(prev_row['abstentions']))
    members_present = set(list(row['ayes']) + list(row['noes']) + list(row['abstentions']))
    if (len(prev_members_present) == 0) or (len(members_present) == 0):
        continue
    new_row = row.copy()
    new_row['days_between'] = days_between
    new_row['frac_new'] = len(prev_members_present - members_present) / len(prev_members_present)
    df.append(new_row)

df = pd.DataFrame(df).reset_index(drop=True)
df['outcome'] = 1
df.loc[df['project_result'].isin(["DELAYED", "DENIED"]), 'outcome'] = 0

df['frac_new_z'] = (df['frac_new'] - df['frac_new'].mean()) / df['frac_new'].std()  # demean
df['atypicality_z'] = (df['atypicality'] - df['atypicality'].mean()) # demean

df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "rehearing_delays.parquet"))

In [34]:
# Running R regressions

res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/19-rehearing-delays.R"], check=True, capture_output=True, text=True)
print(res.stdout)


                                      Dependent variable:            
                           ------------------------------------------
                                            outcome                  
                             (1)     (2)      (3)      (4)      (5)  
---------------------------------------------------------------------
atypicality_z              -0.172   0.060    0.416    -0.408  -0.050 
                           (0.392) (0.477)  (0.547)  (0.455)  (0.407)
                                                                     
frac_new_z                 -0.351   -0.525   -0.563   -0.642  -0.418 
                           (0.365) (0.479)  (0.482)  (0.440)  (0.390)
                                                                     
atypicality_X_frac_new     -0.740* -1.125** -1.010** -0.734*  -0.729*
                           (0.381) (0.483)  (0.465)  (0.433)  (0.395)
                                                                     
log_gap            

In [ ]:
# Output table

header = r"""\begin{table}[H]
\centering
\caption{Factors Influencing Approval or Delay - Re-hearings}
\vspace{0.2cm}
\label{tab_rehearing_delays}
\begin{adjustbox}{max height=0.45\textheight}
\begin{threeparttable}
\begin{tabular}{lccccc}
\multicolumn{6}{l}{Binary Logit (0=Delayed/Denied, 1=Approved/Approved with Conditions)} \\
\toprule
 & (1) & (2) & (3) & (4) & (5) \\
\midrule
 &  &  &  & & \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports coefficient estimates from the binary logit regression described in Section \ref{sec_results}, where the sample is restricted to re-hearings of previously heard cases. Dummies for missing values of height and square footage are included for cases without reported physical dimensions. Consent calendar is not included due to lack of variation.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
reg_names = ["r1b", "r2b", "r3b","r4b"]

vars = [
    ("is_residentialTRUE", "Residential Development"),
    ("is_mixed_useTRUE", "Mixed-Use Development"),
    ("is_nonresidentialTRUE", "Non-Residential Development"),
    ("log_square_footage", "$\\ln$(Square Footage)"),
    ("height", "Height (ft)"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)"),
    ("agenda_order", "Agenda Order"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendarTRUE", "Consent Calendar"),
    ("atypicality", "Atypicality"),
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += "\n & & & & \\\\ \n"

tbl += "Suffix Group Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="sfx_grp_CUPTRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Council District Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cd_1")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Year Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="yr_2019")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Embedding Cluster Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cluster_fe1TRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "\n & & & & \\\\ \n"


tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

tbl += "McFadden's Pseudo-R$^2$"
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="pseudo_r2")
    if idx.sum()==0:
        tbl += " & "
    else:
        tbl += f" & {coefs_df.loc[idx, "estimate"].values[0]:.3f}"
tbl += r" \\ [1.8ex]" + "\n"



table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_blogit_results.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)
